# 课程 4 · Python 记得上一轮，为什么模型未必知道？

课程 3 中，你已经从响应里取出了正文、读懂了 JSON，也知道格式正确不代表内容正确。那里的运行结果来自人工样例。**这次接通真实模型，亲眼观察一次对话是怎样发生的。**

有人咨询「星港观测站开放夜」：先说自己和妹妹想参加，随后补一句“那再带上我 8 岁的弟弟呢？”如果程序明明保存了前面的内容，模型为什么还可能询问人数、年龄或规则？

我们手里有：能运行 Python 的环境、你已有的模型 API 账号、一页人工编写的虚构活动规则。没有报名后台，也没有真实活动订单。我们要得到三次真实回答和一份有依据的实验说明。

**默认流程只调用 3 次**：首次带材料提问 → 只发追问 → 带历史发同一个追问。回答没有预填，也不预设“不带历史一定答错”。主实验约 30 分钟，可分两段做。


## 第一段：拿到真实回答，再试一次追问（约 15 分钟）

选择你界面上的 **llm-from-zero** 内核。下面打印它实际使用的 Python；显示名称不必叫 `.venv`。路径查找、显示结果的辅助代码已经提供，今天不把它们作为填空。


In [ ]:
from pathlib import Path
import sys
import json

candidates = [Path.cwd() / "04-model-api", Path.cwd()]
COURSE_DIR = next((p for p in candidates if (p / "model_client.py").exists()), None)
if COURSE_DIR is None:
    raise RuntimeError("请从项目根目录或 04-model-api 目录运行。")
if str(COURSE_DIR) not in sys.path:
    sys.path.insert(0, str(COURSE_DIR))
from model_client import configure, call_model

experiment_log = []  # 只保存本次内核里的成功结果，不含密钥。

def show_result(label, messages, result):
    experiment_log.append({"label": label, "message_count": len(messages), "result": result})
    print(label)
    print(result["text"])
    print(json.dumps({
        "message_count": len(messages),
        "model": result["model"],
        "usage": result["usage"],
        "elapsed_seconds": result["elapsed_seconds"],
    }, ensure_ascii=False, indent=2))

print("Python:", sys.executable)
print("课程目录:", COURSE_DIR)


### 我们先知道什么？

下面的票价、年龄限制和活动安排都是**人工设计的课堂材料**。读完先自己判断“两名游客是否符合条件”，这样才能核对模型，不能用模型的话来证明模型的话。


In [ ]:
rules = (COURSE_DIR / "data" / "observatory_rules.md").read_text(encoding="utf-8")
print(rules)


### 选择已有的服务

`configure()` 只读取配置，不发请求。默认 `RUN_LIVE=False` 会明确停下；不会回退到人工回答。准备好后改成 `True`，再运行配置格。

- DeepSeek：`MODEL=None` 时默认用 `deepseek-flash`，也可以填写账号可用的模型 ID。
- TokenHub：把 `PROVIDER` 改成 `"tokenhub"`，在 `MODEL` 填控制台已开通的 **DeepSeek 系列** ID；没填时会询问。按量 API 与 Token Plan 编程套餐不同。
- 缺少环境变量时，会在隐藏输入框中询问密钥。密钥仅保留在内核内存，不写进 notebook；配置的显示内容不包含密钥。地域和错误排查见 [README](README.md)。

每运行一次调用格，就可能产生一次费用。请求不自动重试。使用非思考、非流式模式，`temperature=0`，输出上限 2048 tokens；**低温不保证每次回答严格相同**。


In [ ]:
RUN_LIVE = False  # 确认要接通真实服务后，手动改为 True。
PROVIDER = "deepseek"  # 或 "tokenhub"
MODEL = None  # TokenHub 请填已开通的 DeepSeek 系列模型 ID。

if not RUN_LIVE:
    raise RuntimeError("尚未启用真实调用。准备好后将 RUN_LIVE 改为 True；没有生成任何回答。")
config = configure(provider=PROVIDER, model=MODEL)
print(config)


### 第一次：把规则和问题一起发出

`system` 写这次任务的一般要求，`user` 带活动材料和咨询内容。消息的字典写法你已经学过，这里直接给出。`call_model(messages, config)` 把它们发送给选定服务，并整理返回值：

- `text`：真实响应的 `choices[0].message.content`。
- `usage`：服务返回的 token 用量；没有返回就保留缺失，不能当作 0。
- `model`、`finish_reason`、`elapsed_seconds`：用于记录这次调用。

**先口头预测**：26 岁的林舟和 16 岁的阿岚能否参加露台观测？票价多少？是哪两条规则支持你的判断？


In [ ]:
system_text = (
    "你是活动咨询助手。只依据当前消息中提供的活动规则和访客信息回答，"
    "缺少信息就明确指出并询问，不编造规则、不声称已经报名。"
    "涉及票价和活动安排时引用规则编号；用简洁中文回答，尽量不超过180字。"
)
first_question = "我是26岁的林舟，准备带16岁的妹妹阿岚参加露台观测。我们两人能参加吗？总票价多少？"
first_user_content = f"活动材料：\n{rules}\n咨询：{first_question}"
first_messages = [
    {"role": "system", "content": system_text},
    {"role": "user", "content": first_user_content},
]
print("发出的角色:", [message["role"] for message in first_messages])
print("发出的用户内容:", first_messages[-1]["content"])


In [ ]:
if not RUN_LIVE:
    raise RuntimeError("RUN_LIVE 未开启；没有请求，也没有替代回答。")
first_result = call_model(first_messages, config)  # 真实调用 1
show_result("A：带规则的首次提问", first_messages, first_result)


**我的第一次核对**（先核对，再继续；即使有错也如实记）：

- 模型给的参加条件和票价：____
- 我依据哪条原始规则判断，哪里对／哪里不对：____

返回文字说明了什么，和它是否真的执行了报名，是两件事。本课没有连接报名系统。


### 第二次：Python 有变量，但这次请求里有什么？

`first_result` 仍保存在 Python 中。现在我们只发送相同的一般要求和一句追问，**不把上一轮的规则、问题和回答放入消息列表**。

这一步是对照实验。不要先认定模型会答错：它也可能主动追问，或给一个合理但缺乏本活动依据的通用建议。我们要看它实际拿到了什么、说了什么。


In [ ]:
followup_question = "那再带上我8岁的弟弟小禾呢？费用和安排需要怎么改？"
no_history_messages = [
    {"role": "system", "content": system_text},
    {"role": "user", "content": followup_question},
]
print("Python 仍保存首次回答:", bool(first_result["text"]))
print("第二次实际发送的消息:")
print(json.dumps(no_history_messages, ensure_ascii=False, indent=2))


In [ ]:
if not RUN_LIVE:
    raise RuntimeError("RUN_LIVE 未开启；没有请求，也没有替代回答。")
no_history_result = call_model(no_history_messages, config)  # 真实调用 2
show_result("B：只发送追问", no_history_messages, no_history_result)


**我的第二次观察**：

- 它追问了什么、猜测了什么，或直接给出了什么结论？引用一句：____
- 单看第二次发送的消息，是否包含原先的人数、年龄和票价规则？____
- 如果某个数字碰巧正确，我现在能否证明它取用了第一轮历史？为什么？____

可以在这里停下来讨论，再做第二段。保留当前内核就能继续使用变量；如果重启内核，需要重新准备前面的结果，重跑调用会再次计费。


## 第二段：把模型需要知道的内容真正发出去（约 15 分钟）

### Python 变量与请求消息是两回事

这份客户端使用普通 Chat Completions 请求，没有传服务端会话 ID。服务收到的是这次请求的 `messages`，它不会自动读取你电脑里的 `first_result`。同一个 API key 和同一个模型名，也不会替我们把聊天变量放进请求。

要让第三次回答参考上一轮，准备这个有顺序的列表：

| 顺序 | role | content 来自哪里 |
|---|---|---|
| 1 | system | 原来的一般要求 |
| 2 | user | 首次提交的完整材料和问题 |
| 3 | assistant | 第一次实际得到的回答正文 |
| 4 | user | 与第二次完全相同的追问 |

这不是重新训练模型，而是把已有信息作为本次输入。其他支持服务端会话管理的产品可能代管历史；这里讨论的是**当前使用的接口和程序**。


### TODO 1：构造完整历史

把 `history_messages` 写成上表对应的消息列表。使用已有的 `first_messages`、`first_result`、`followup_question`；不要重新编一份“模型回答”，也不要修改 `first_messages`。

你可以创建新列表并添加消息，也可以使用列表相加。`first_messages + [...]` 会得到新列表；`first_messages.append(...)` 会修改原来的列表。这里需要保留首次请求用于对照。


In [ ]:
# TODO 1：构造第三次请求，包含原请求、真实 assistant 回答和当前追问。
history_messages = None

assert isinstance(history_messages, list), "先完成 TODO 1：创建消息列表。"
assert [m["role"] for m in history_messages] == ["system", "user", "assistant", "user"]
assert history_messages[:2] == first_messages and len(first_messages) == 2
assert history_messages[2]["content"] == first_result["text"]
assert history_messages[-1] == no_history_messages[-1]
print("第三次角色顺序:", [m["role"] for m in history_messages])
print("历史已包含真实回答；首次请求仍保留两条消息。")


In [ ]:
if not RUN_LIVE:
    raise RuntimeError("RUN_LIVE 未开启；没有请求，也没有替代回答。")
history_result = call_model(history_messages, config)  # 真实调用 3
show_result("C：带历史发送相同追问", history_messages, history_result)


### 看结果，不替模型下预设结论

这次需要同时处理：新增年龄、三人的总票价、露台年龄限制、低龄游客陪同要求。**规则才是核对依据，先前的 assistant 回答也可能有错。** 把错误回答放回历史，并不会把它变成事实。

填写下面的记录；如果模型答错，保留原文并解释错在哪里。如果 B、C 都回答合理，也照实写，不需要制造失败。

| 核对项 | B：只发追问 | C：带历史追问 | 你的判断依据 |
|---|---|---|---|
| 是否清楚此前是哪两人 | ____ | ____ | 首次问题 |
| 总票价是否有资料依据 | ____ | ____ | R1 |
| 是否处理 8 岁游客不能上露台 | ____ | ____ | R2 |
| 安排是否满足成人全程陪同 | ____ | ____ | R3、R4 |
| 是否把建议误说成已报名 | ____ | ____ | R5 |

消息列表能直接证明“我们发出了哪些信息”。仅凭一次回答，不能证明模型每次都能正确使用它们，也不能证明所有多轮产品都必须由用户手工保存历史。


### 带历史还有一个可测量的代价：它也占输入 token

你以前用过费用公式。这次换成真实返回的 `usage`，但单价仍统一采用**教学假设**：输入 **3 元 / 百万 tokens**，输出 **9 元 / 百万 tokens**。这不是 DeepSeek 或 TokenHub 的实际报价。

一次请求费用：

`输入 tokens × 3 元 / 1,000,000 tokens + 输出 tokens × 9 元 / 1,000,000 tokens`

几次请求组成一个任务，就把每次费用相加。注意区分两种范围：

- **两轮对话任务** = A（首次提问）+ C（带历史追问）。
- **三次课堂实验** = A + B（独立对照）+ C。

C 的 `prompt_tokens` 已经包含本次发送的规则、先前问答和追问。**不要在 C 的用量之外再加一次历史 token**；A 自己那次调用仍然要计费。缺少用量是“未知”，不是 0。


In [ ]:
two_turn_results = [first_result, history_result]
lesson_results = [first_result, no_history_result, history_result]
for label, result in zip(["A：首次", "B：对照", "C：带历史"], lesson_results):
    print(label, result["usage"])
print("本次内核记录的成功调用数:", len(experiment_log))


### TODO 2：计算指定任务的费用

函数接收若干次调用结果。缺少输入或输出用量时，下面提供的检查会返回 `None`，表示无法核算。你负责剩下的循环、累加和单位换算；可以沿用以前写循环的方式。

这里的 `result.get("usage", {})` 表示：有 `usage` 就取出来，没有就返回空字典；`"prompt_tokens" not in usage` 检查这个键是否缺失。先判断键存在，再取数值，就不会把“没返回”误当作“用了 0 个”。

先写公式再运行。计算结果只是**这些成功记录按假设价格的估算**。重跑、超时、截断等可能产生未计入的费用；真实账单还与服务商价格和缓存命中有关，要到控制台核对。


In [ ]:
def estimate_task_cost(results, input_per_million=3.0, output_per_million=9.0):
    for result in results:
        usage = result.get("usage", {})
        if "prompt_tokens" not in usage or "completion_tokens" not in usage:
            return None  # 用量未知，不能估成免费。
    # TODO 2：逐次计算并相加，返回单位为“元”的总费用。
    total_yuan = None
    assert total_yuan is not None, "TODO 2 尚未完成：请计算并返回总费用。"
    return total_yuan

for name, results in [("两轮对话任务 A+C", two_turn_results), ("三次课堂实验 A+B+C", lesson_results)]:
    estimated = estimate_task_cost(results)
    if estimated is None:
        print(name, "：无法核算；服务没有返回完整 usage，不能记为 0。")
    else:
        print(f"{name}：{estimated:.6f} 元（教学假设价）")


**我的费用记录**（每项单独填写）：

- 我使用的公式，含单位：____
- A+C 的估算：____ 元；A+B+C 的估算：____ 元。
- C 的输入用量包含什么，为什么不再额外加算历史：____
- 这份记录有哪些不能替代真实账单的地方：____

这里的两个列表各取三次实验的当前结果。重新运行某个调用格会覆盖相应变量，但旧调用仍可能计费；不能用这两个列表声称已经核清账号总账。


### 用一分钟解释给一位同事听

按四个独立答题槽写，不需要堆术语。回答围绕“为什么 Python 有上一轮变量，模型却未必知道？”

1. **结论**：对当前接口，你选择怎样组织多轮输入？____
2. **机制**：Python 保存变量和 HTTP 请求传消息之间是什么关系？____
3. **实验记录**：引用 B、C 各一句实际回答，再说明哪项原始规则能验证它们。____
4. **限制**：这三次请求不能证明什么？带历史是否保证正确、是否免费？____

如果效果没达到预期，也能完成本课：保留真实记录、定位缺少的信息或错误依据，比把结果写成预想答案更有价值。

完成后进入 [homework.ipynb](homework.ipynb)：独立把消息构造成函数，换一组数字算费，并整理一次面试式解释。作业不要求新增 API 调用。
